In [1]:
from transformers import AutoTokenizer, BitsAndBytesConfig, TextStreamer 
from liger_kernel.transformers import AutoLigerKernelForCausalLM as AutoModelForCausalLM
import torch
import time

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

#MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# Initialize quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_use_double_quant=True, 
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config
)


# Create text streamer
streamer = TextStreamer(tokenizer)

# Prepare input
messages = [
    {"role": "user", "content": "Write me a poem about Machine Learning."},
]
input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

# Generate with timing
start_time = time.time()
with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        streamer=streamer
    )
end_time = time.time()

# Calculate and print token speed
generation_time = end_time - start_time
num_new_tokens = len(outputs[0]) - len(input_ids[0])
tokens_per_second = num_new_tokens / generation_time

print(f"\nGeneration Stats:")
print(f"Total tokens generated: {num_new_tokens}")
print(f"Generation time: {generation_time:.2f} seconds")
print(f"Speed: {tokens_per_second:.2f} tokens/second")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Write me a poem about Machine Learning.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

In realms of code and circuitry bright,
A world of wonder, born of data's light,
Machine Learning, a magic spell,
That weaves insights from the digital shell.

With algorithms that dance and play,
The machines learn, night and endless day,
From inputs vast, they craft and shape,
Predictions born, from patterns they escape.

The neural nets, a labyrinthine maze,
Where weights and biases, in harmony sway,
The gradient descent, a winding stair,
That guides the models, with precision fair.

The regression lines, a gentle slope,
The classification trees, a branching hope,
The clustering groups, a fuzzy sphere,
The decision trees, a branching lair.

In this domain, of math and art,
The machines learn, with a curious heart,
F